In [14]:
import anthropic
import pandas as pd
import re
import time
from pathlib import Path
import os

In [ ]:
API_KEY = ""
MODEL = "claude-opus-4-5"
DELAY_BETWEEN_CALLS = 1.0

In [16]:
def load_speech(filepath):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Speech file not found: {filepath}")
    if path.suffix.lower() != ".txt":
        raise ValueError(f"Expected a .txt file, got: {path.suffix}")
    
    raw = path.read_text(encoding="utf-8", errors="replace")
    text = raw.replace("\r\n", "\n").replace("\r", "\n")
    test = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [17]:
def build_prompt(text: str) -> str:
    return f"""You are a political language analyst. Score the following text using this rubric:

Count all instances where the text portrays working-class or low-income people as capable, hardworking, independent, responsible, or deserving of opportunity. This includes language emphasizing effort, contribution, dignity, self-reliance, or upward mobility. Each instance counts as one point toward the dignity score.

Count all instances where the text portrays these groups as underprivileged, marginalized, struggling working families, helpless, dependent, vulnerable, struggling, or in need of rescue or assistance. This includes language emphasizing hardship without agency, need for aid, or reliance on external support. Each instance counts as one point toward the pity score.

Do not have overlap between the two scores. Do not count generic references like "people" unless they clearly refer to a specific subgroup.

Do not count generic references like “people” unless they clearly refer to a subgroup group.
Do not explain your reasoning. Do not list examples. Do not use headers or bullet points.
Your entire response must be exactly one line in this format:
Dignity: X, Pity: Y

Text to score:
\"\"\"{text}\"\"\"
"""


def call_api(text):
    client = anthropic.Anthropic(api_key=API_KEY)
    prompt = build_prompt(text)

    message = client.messages.create(
        model=MODEL,
        max_tokens=128,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

In [18]:
def parse_response(raw: str) -> dict:
    pattern = re.search(
        r"Dignity:\s*(\d+),\s*Pity:\s*(\d+)",
        raw, re.IGNORECASE
    )

    return {
        "dignity":      int(pattern.group(1)) if pattern else None,
        "pity":         int(pattern.group(2)) if pattern else None,
        "raw_response": raw,
        "parse_error":  None if pattern else "Could not parse Dignity/Pity"
    }

def score_speech(speech_path: str) -> dict:
    speech = load_speech(speech_path)
    raw = call_api(speech)
    result = parse_response(raw)
    result["filename"] = Path(speech_path).name
    return result

In [19]:
def score_all_speeches(speech_folders, output_path):
    results = []

    for folder in speech_folders:
        for filename in os.listdir(folder):
            if filename.endswith(".txt"):
                speech_path = os.path.join(folder, filename)
                print(f"Scoring: {speech_path}")
                result = score_speech(speech_path)
                results.append(result)
                time.sleep(DELAY_BETWEEN_CALLS)

    df = pd.DataFrame(results)
    df.to_excel(output_path, index=False)
    print(f"\nDone! Results saved to {output_path}")
    return df

folders = [
    "Zara-Ahsan-Thesis-Repository/speeches/andy beshear",
    "Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown",
    "Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker",
    "Zara-Ahsan-Thesis-Repository/speeches/Kris Kobach",
    "Zara-Ahsan-Thesis-Repository/speeches/Larry Hogan",
    "Zara-Ahsan-Thesis-Repository/speeches/Laura Kelly",
    "Zara-Ahsan-Thesis-Repository/speeches/Martha Coakley",
    "Zara-Ahsan-Thesis-Repository/speeches/matt bevin",
    "Zara-Ahsan-Thesis-Repository/speeches/Phil Scott",
    "Zara-Ahsan-Thesis-Repository/speeches/Sue Minter"
]

df_results = score_all_speeches(
    speech_folders=folders,
    output_path="pity_party_scores.xlsx"
)

Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\andy_beshear_final_debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\andy_beshear_partisan_winning_speech.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\andy_beshear_tweets.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/andy beshear\Fancy Farms.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown\Fox interview.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown\Oct 7 guber debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\MACDC's 2014 Convention.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\Oct 21 Gubernatorial debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\Oct 23, 2014 Massachusetts Gubernatorial Debate.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker\The Berkshire Eagle.txt
Scoring: Zara-Ahsan-Thesis-Repository/speeches/Kris Kobach\2018 Kansas Governor Debate - September 5, 2018